# Week 7 — Optimization

**Focus:** How do optimization and unsupervised representations behave in noisy, redundant data?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week07/week7_demos.ipynb)

## What this notebook is about

Many financial ML problems are hard not because the objective is mysterious, but because the data are noisy, redundant, and badly conditioned. This notebook asks how optimization and unsupervised representations behave under those conditions.

You will compare gradient-based optimizers on an ill-conditioned surface, watch small input errors become large portfolio changes, audit search budgets and future invariance, compress a factor panel with PCA, compare sample eigenvalues with a random-matrix noise band, and use clustering to organize assets and candidate regimes.

By the end, you should be able to:

- explain why scaling and conditioning affect optimization;
- distinguish numerical convergence from economic correctness;
- read a principal component as an estimated direction rather than a named truth;
- explain what an eigenvalue noise band is screening for; and
- treat clusters as descriptions whose stability and decision value still require testing.

The simulations make recovery possible or impossible by design. The industry-data companion shows what the same tools look like without supplying known labels.

### Course context

For a concise review and the optional textbook route, see [Week 7 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week07/week7_summary.md) and [Week 7 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week07/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — Convex and non-convex objective geometry — <a href="#Demo-1-%E2%80%94-Convex-and-non-convex-objective-geometry">Jupyter</a> · <a href="#scrollTo=md-965ed26e7d57">Colab</a></li>
<li>Demo 2 — Gradient descent vs. momentum vs. Adam on an ill-conditioned bowl — <a href="#Demo-2-%E2%80%94-Gradient-descent-vs.-momentum-vs.-Adam-on-an-ill-conditioned-bowl">Jupyter</a> · <a href="#scrollTo=md-09006b9e5185">Colab</a></li>
<li>Demo 3 — Search budget and development overfitting — <a href="#Demo-3-%E2%80%94-Search-budget-and-development-overfitting">Jupyter</a> · <a href="#scrollTo=md-e3963dd4248d">Colab</a></li>
<li>Demo 4 — Stress a tiny mean-variance optimum — <a href="#Demo-4-%E2%80%94-Stress-a-tiny-mean-variance-optimum">Jupyter</a> · <a href="#scrollTo=md-0338f3db3208">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 5 — PCA on a simulated factor panel — <a href="#Demo-5-%E2%80%94-PCA-on-a-simulated-factor-panel">Jupyter</a> · <a href="#scrollTo=md-b44785d63714">Colab</a></li>
<li>Demo 6 — Future-invariance test for PCA — <a href="#Demo-6-%E2%80%94-Future-invariance-test-for-PCA">Jupyter</a> · <a href="#scrollTo=md-e1fe84657a0c">Colab</a></li>
<li>Real-data companion — PCA on twelve industry portfolios — <a href="#Real-data-companion-%E2%80%94-PCA-on-twelve-industry-portfolios">Jupyter</a> · <a href="#scrollTo=md-de33b3e667e3">Colab</a></li>
<li>Demo 7 — Marchenko-Pastur eigenvalue band — <a href="#Demo-7-%E2%80%94-Marchenko-Pastur-eigenvalue-band">Jupyter</a> · <a href="#scrollTo=md-7ea2c9043d5b">Colab</a></li>
<li>Demo 8 — Cluster a correlation matrix to recover sectors — <a href="#Demo-8-%E2%80%94-Cluster-a-correlation-matrix-to-recover-sectors">Jupyter</a> · <a href="#scrollTo=md-e118236be73e">Colab</a></li>
<li>Demo 9 — Frozen-centroid regime assignment — <a href="#Demo-9-%E2%80%94-Frozen-centroid-regime-assignment">Jupyter</a> · <a href="#scrollTo=md-ad14ad4926d2">Colab</a></li>
</ul>

## Project research-record checkpoint

Your project record remains the single `RESEARCH_RECORD.md` at the project-repository root; this notebook only supplies a checkpoint. After this week, update whichever of these fields actually apply:

1. **Fitted state:** transformations or representations, fit sample and cutoff, optimizer or solver, and version.
2. **Search ledger:** candidate representations, hyperparameters or cluster counts, selection rule, and total trials.
3. **Stability:** conditioning, perturbation, resampling, or later-period evidence that could overturn the choice.
4. **Baseline:** the simpler feasible representation or allocation against which the added machinery is evaluated.
5. **Interpretation limit:** what an estimated component, cluster, or regime does not establish.

If a choice changes after seeing development evidence, append a dated decision-log entry; do not silently rewrite the earlier plan.

### Six recorded-lecture mini-projects

1. **Classify geometry — Demo 1:** use the chord test to distinguish a convex objective from a constructive non-convex counterexample.
2. **Make descent crawl — Demo 2:** compare optimizer paths on ill-conditioned geometry and alter the scaling.
3. **Cap the search — Demo 3:** hold the candidate sequence fixed and compare development with untouched loss at three budgets.
4. **Stress the optimum — Demo 4:** perturb means and covariance and record weight movement, turnover, and objective regret.
5. **Audit a PCA representation — Demos 5–7 plus the real-data PCA companion:** interpret a planted factor, verify future invariance, compare eigenvalues with a noise band, and test a frozen component on industry returns.
6. **Recover groups, then make a regime prospective — Demos 8–9:** recover planted asset sectors, then freeze training-period centroids and compare later assignments with retrospective full-sample labels.

In [ ]:
# Setup: install packages not preinstalled in Colab.
%pip install -q finmlsim

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.linalg import eigh

from finmlsim import simulate as sim

rng = np.random.default_rng(42)


def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url


def save_lecture_figure(fig, filename):
    """Save a notebook figure when the course repository is available."""
    for root in (Path.cwd(), *Path.cwd().parents):
        output = root / "lectures" / "week07" / "figures"
        if output.exists():
            target = output / filename
            fig.savefig(target, dpi=180, bbox_inches="tight", facecolor="white")
            print(f"saved lecture figure: {target}")
            return

### Demo 1 — Convex and non-convex objective geometry

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
x_grid = np.linspace(-1.5, 1.5, 400)

def convex_objective(x):
    return x ** 2

def nonconvex_objective(x):
    return (x ** 2 - 1) ** 2

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
examples = [
    (convex_objective, (-1.2, 1.0), "convex objective"),
    (nonconvex_objective, (-1.15, 1.15), "non-convex objective"),
]
for ax, (objective, (left, right), title) in zip(axes, examples):
    chord_x = np.linspace(left, right, 150)
    chord_y = (objective(left) +
               (objective(right) - objective(left)) *
               (chord_x - left) / (right - left))
    gap = chord_y - objective(chord_x)
    ax.plot(x_grid, objective(x_grid), lw=2.4, label="objective")
    ax.plot(chord_x, chord_y, "--", lw=2.0, label="chord")
    ax.scatter([left, right], [objective(left), objective(right)], zorder=3)
    ax.set_title(title)
    ax.set_xlabel("decision variable")
    ax.set_ylabel("objective")
    ax.legend(frameon=False)
    print(f"{title}: minimum(chord - objective) = {gap.min():+.3f}")
fig.suptitle("A convex objective stays below every chord; a non-convex one need not")
fig.tight_layout()
save_lecture_figure(fig, "demo1_convex_nonconvex.png")
plt.show()

print("A negative chord gap is a constructive failure of convexity.")
print("This geometric test says nothing about whether the objective is economically well specified.")

### Mini-project: classify the problem

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

For four objectives, identify the variable, feasible set, convexity,
and whether the solution is unique.

**Notebook:** run **Demo 1 — Convex and non-convex objective geometry**, then classify the four problems.

| Problem | Variable | Feasible set | Convex? | Unique solution? |
|---|---|---|---|---|
| Ridge regression | | | | |
| Lasso regression | | | | |
| Long-only mean-variance | | | | |
| Integer lot sizes | | | | |

**Research record:** for each of the four objectives: the variable chosen, the feasible set, whether the problem is convex, and whether the solution is unique.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 2 — Gradient descent vs. momentum vs. Adam on an ill-conditioned bowl

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
# An elongated quadratic: f(w) = 0.5 * (w1^2 + 25 * w2^2)
def f(w):
    return 0.5 * (w[0] ** 2 + 25 * w[1] ** 2)

def grad(w):
    return np.array([w[0], 25 * w[1]])

def run(opt, eta=0.05, T=80):
    w = np.array([1.0, 1.0])
    m = np.zeros(2)
    v = np.zeros(2)
    path = [w.copy()]
    for t in range(1, T + 1):
        g = grad(w)
        if opt == "gd":
            w = w - eta * g
        elif opt == "momentum":
            m = 0.9 * m + g
            w = w - eta * m
        elif opt == "adam":
            m = 0.9 * m + 0.1 * g
            v = 0.999 * v + 0.001 * g ** 2
            mhat = m / (1 - 0.9 ** t)
            vhat = v / (1 - 0.999 ** t)
            w = w - 0.3 * mhat / (np.sqrt(vhat) + 1e-8)
        path.append(w.copy())
    return np.array(path)

paths = {name: run(name) for name in ("gd", "momentum", "adam")}
fig, ax = plt.subplots(figsize=(6, 4))
x1, x2 = np.meshgrid(np.linspace(-1.15, 1.15, 180), np.linspace(-1.15, 1.15, 180))
ax.contour(x1, x2, 0.5 * (x1 ** 2 + 25 * x2 ** 2), levels=18,
           colors="#c7c7c7", linewidths=.7)
for name, p in paths.items():
    ax.plot(p[:, 0], p[:, 1], "-o", ms=2, label=name)
ax.plot(0, 0, "k*", ms=12, label="optimum")
ax.set_title("Three optimizers on an elongated bowl")
ax.set(xlabel="parameter 1", ylabel="parameter 2")
ax.legend(frameon=False)
fig.tight_layout()
save_lecture_figure(fig, "demo2_optimizer_paths.png")
plt.show()

### Mini-project: make gradient descent crawl

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Rescale one feature, hold the step rule fixed, and plot the path. Then
standardize inside the training boundary and repeat.

**Notebook:** run **Demo 2 — Gradient descent on an ill-conditioned bowl** and modify its scaling.

**Research record:** the feature rescaled and by how much; the fixed step rule; the path before and after standardizing inside the training boundary.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 3 — Search budget and development overfitting

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
search_rng = np.random.default_rng(1)
n_candidates = 500
theta = search_rng.uniform(-2, 2, n_candidates)
population_loss = 0.20 + (theta - 0.70) ** 2
development_loss = population_loss + search_rng.normal(0, 0.30, n_candidates)
untouched_loss = population_loss + search_rng.normal(0, 0.05, n_candidates)
budgets = np.array([20, 100, 500])
selected = [np.argmin(development_loss[:budget]) for budget in budgets]
best_development = np.array([development_loss[index] for index in selected])
selected_untouched = np.array([untouched_loss[index] for index in selected])

fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.plot(budgets, best_development, "o-", lw=2.2, label="best development loss")
ax.plot(budgets, selected_untouched, "o-", lw=2.2,
        label="untouched loss of selected candidate")
ax.set(xlabel="candidate evaluations", ylabel="loss",
       title="A larger search can fit development noise")
ax.legend(frameon=False); ax.grid(alpha=.2)
fig.tight_layout()
save_lecture_figure(fig, "demo3_search_budget.png")
plt.show()

for budget, index in zip(budgets, selected):
    print(f"budget {budget:3d}: theta={theta[index]:+.3f}, "
          f"development={development_loss[index]:+.3f}, "
          f"untouched={untouched_loss[index]:+.3f}")
print("The budgets and random-search sequence were fixed before the untouched losses were inspected.")

### Mini-project: cap the search

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run the same derivative-free optimizer at three evaluation budgets.
Plot best development objective and untouched objective against budget.

**Notebook:** run **Demo 3 — Search budget and development overfitting**.

**Research record:** the three evaluation budgets; best development objective at each; the untouched objective at each; the gap between them.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — Stress a tiny mean-variance optimum

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
# Show that plausible perturbations to both mu and Sigma can move w = inv(Sigma) @ mu.
K = 12
rho = 0.84
Sigma = rho * np.ones((K, K)) + (1 - rho) * np.eye(K)
mu = rng.standard_normal(K) * 0.02
stress_rng = np.random.default_rng(704)
mean_direction = stress_rng.standard_normal(K)
cov_direction = stress_rng.standard_normal((K, K))
cov_direction = cov_direction @ cov_direction.T / K
stress_levels = np.array([0.00, 0.05, 0.10, 0.20])

def mv_weights(mean, covariance):
    return np.linalg.solve(covariance, mean)

def mv_objective(weights, mean, covariance):
    return 0.5 * weights @ covariance @ weights - mean @ weights

base = mv_weights(mu, Sigma)
stressed_weights, instability, turnover, regret = [], [], [], []
for level in stress_levels:
    mean_i = mu + level * mu.std() * mean_direction
    covariance_i = Sigma + level * cov_direction
    weights_i = mv_weights(mean_i, covariance_i)
    stressed_weights.append(weights_i)
    instability.append(np.linalg.norm(weights_i - base))
    turnover.append(np.abs(weights_i - base).sum())
    regret.append(mv_objective(base, mean_i, covariance_i) -
                  mv_objective(weights_i, mean_i, covariance_i))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, values, title in zip(
        axes, [instability, turnover, regret],
        ["weight instability", "turnover from base", "objective regret of base"]):
    ax.plot(stress_levels, values, "o-", lw=2.1)
    ax.set(xlabel="declared perturbation scale", title=title)
    ax.grid(alpha=.2)
fig.suptitle("Demo 4: nearby inputs can imply different portfolio actions")
fig.tight_layout(rect=(0, 0, 1, .91))
save_lecture_figure(fig, "demo4_portfolio_stress.png")
plt.show()

for level, move, trade, loss in zip(stress_levels, instability, turnover, regret):
    print(f"stress {level:.0%}: L2 move={move:.3f}, turnover={trade:.3f}, regret={loss:.5f}")

### Mini-project: stress the optimum

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Perturb expected returns and covariance within a declared range. Plot
weight instability, turnover, and the change in the modeled objective.

**Notebook:** run **Demo 4 — Stress a tiny mean-variance optimum**.

**Research record:** the declared perturbation range for means and covariance; weight instability, turnover, and the modeled-objective change across it.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 5 — PCA on a simulated factor panel

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
panel = sim.panel(n=1500, n_assets=60, market_vol=0.01, idio_vol=0.015, seed=7)
# sim.panel returns DataFrame or array of returns; standardize
R = np.asarray(panel)
R = R - R.mean(0)

pca = PCA(n_components=10).fit(R)
print("Explained variance ratios (top 10):")
print(pca.explained_variance_ratio_.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(range(1, 11), pca.explained_variance_ratio_[:10])
axes[0].set_title("Scree plot — one tall bar, then scree")
axes[0].set_xlabel("Component")

# PC1 loadings
v1 = pca.components_[0]
if v1.mean() < 0:
    v1 = -v1
axes[1].bar(range(len(v1)), v1)
axes[1].set_title("PC1 loadings — same sign across the cross-section")
axes[1].set_xlabel("asset")
fig.suptitle("Demo 5: one planted common factor dominates the panel")
fig.tight_layout(rect=(0, 0, 1, .91))
save_lecture_figure(fig, "demo5_pca_factor_panel.png")
plt.show()

# PC1 score vs equal-weighted "market"
score1 = R @ v1
mkt = R.mean(axis=1)
corr = np.corrcoef(score1, mkt)[0, 1]
print(f"corr(PC1 score, EW market) = {corr:+.4f}  (expect ~0.99+)")

### Demo 6 — Future-invariance test for PCA

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
pca_rng = np.random.default_rng(706)

def rotated_cloud(n, angle_degrees):
    base_cloud = pca_rng.standard_normal((n, 2)) @ np.diag([1.8, 0.45])
    angle = np.deg2rad(angle_degrees)
    rotation = np.array([[np.cos(angle), -np.sin(angle)],
                         [np.sin(angle),  np.cos(angle)]])
    return base_cloud @ rotation.T

early = rotated_cloud(220, 20)
later = rotated_cloud(180, 70)
later_perturbed = later @ np.array([[0, -1], [1, 0]]).T

early_fit = PCA(n_components=1).fit(early)
frozen_scores_before = early_fit.transform(early)[:, 0]
frozen_scores_after = early_fit.transform(early)[:, 0]
global_before = PCA(n_components=1).fit(np.vstack([early, later]))
global_after = PCA(n_components=1).fit(np.vstack([early, later_perturbed]))

def aligned_scores(model, observations, reference):
    scores = model.transform(observations)[:, 0]
    return scores if np.corrcoef(scores, reference)[0, 1] >= 0 else -scores

global_scores_before = aligned_scores(global_before, early, frozen_scores_before)
global_scores_after = aligned_scores(global_after, early, frozen_scores_before)
print("largest protected-score change, frozen fit:",
      f"{np.max(np.abs(frozen_scores_after - frozen_scores_before)):.3e}")
print("largest protected-score change, global refit:",
      f"{np.max(np.abs(global_scores_after - global_scores_before)):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].scatter(early[:, 0], early[:, 1], s=12, alpha=.35, label="early")
axes[0].scatter(later[:, 0], later[:, 1], s=12, alpha=.25, label="later")
axes[0].set_title("original later covariance")
axes[1].scatter(early[:, 0], early[:, 1], s=12, alpha=.35, label="same early rows")
axes[1].scatter(later_perturbed[:, 0], later_perturbed[:, 1], s=12, alpha=.25,
                label="changed later covariance")
axes[1].set_title("only the future block changed")
for ax in axes:
    ax.legend(frameon=False); ax.set_aspect("equal")
fig.suptitle("Demo 6: changing only later covariance rotates a global PCA fit")
fig.tight_layout(rect=(0, 0, 1, .91))
save_lecture_figure(fig, "demo6_pca_future_invariance.png")
plt.show()

### Mini-project: rotate the future

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Fit PCA before a cutoff. Change only later covariance and recompute a
global PCA. Confirm that earlier scores move under the global fit and
remain fixed under the training-only fit.

**Notebook:** run **Demo 6 — Future-invariance test for PCA**.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Real-data companion — PCA on twelve industry portfolios

> **Optional empirical comparison:** Run this after the known-truth demo. Use the same transformation and compare the simulation with the historical series. Record one important difference and the most likely reason for it.

In [ ]:
industry_pca_raw, industry_pca_source = course_csv("datasets/famafrench/ff_12industry_daily.csv")
industry_pca_raw["date"] = pd.to_datetime(
    industry_pca_raw["date"].astype(str), format="%Y%m%d"
)
industry_pca_returns = industry_pca_raw.set_index("date").sort_index().loc["1990":] / 100.0
industry_pca_cut = int(0.7 * len(industry_pca_returns))
industry_pca_train = industry_pca_returns.iloc[:industry_pca_cut]
industry_pca_test = industry_pca_returns.iloc[industry_pca_cut:]
industry_pca_mean = industry_pca_train.mean()
industry_pca_scale = industry_pca_train.std()
industry_pca_model = PCA().fit((industry_pca_train - industry_pca_mean) / industry_pca_scale)
industry_pca_score = industry_pca_model.transform(
    (industry_pca_test - industry_pca_mean) / industry_pca_scale
)[:, 0]
industry_pca_loading = industry_pca_model.components_[0].copy()
industry_equal_weight = industry_pca_test.mean(axis=1).to_numpy()
if np.corrcoef(industry_pca_score, industry_equal_weight)[0, 1] < 0:
    industry_pca_score *= -1
    industry_pca_loading *= -1
industry_market_correlation = np.corrcoef(industry_pca_score, industry_equal_weight)[0, 1]

print("Source: Kenneth R. French Data Library, bundled 12 industry portfolios")
print(f"File: {industry_pca_source}")
print(f"Train ends {industry_pca_train.index.max().date()}; frozen test ends {industry_pca_test.index.max().date()}")
print("Units: decimal daily returns, standardized with training-period means and scales")
print("First five explained-variance shares:", np.round(industry_pca_model.explained_variance_ratio_[:5], 3))
print(f"Frozen-test corr(PC1 score, equal-weight industry return): {industry_market_correlation:+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(np.arange(1, 13), industry_pca_model.explained_variance_ratio_)
axes[0].set(xlabel="component", ylabel="training variance share", title="Real industry scree plot")
axes[1].bar(industry_pca_returns.columns, industry_pca_loading)
axes[1].tick_params(axis="x", rotation=60)
axes[1].set(title="PC1 loads broadly across industries", ylabel="loading")
fig.suptitle("Real-data companion: training-period PCA on industry returns")
fig.tight_layout(rect=(0, 0, 1, .91))
save_lecture_figure(fig, "pca_industry_companion.png")
plt.show()
print("The market-like component is an interpretation supported by the loadings and score correlation, not a label supplied to PCA.")

### Demo 7 — Marchenko-Pastur eigenvalue band

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
N, T = 80, 300
X = rng.standard_normal((T, N))      # pure noise
X = (X - X.mean(0)) / X.std(0)
C = (X.T @ X) / T                    # sample correlation
w = np.sort(np.linalg.eigvalsh(C))[::-1]
q = N / T
lam_plus = (1 + np.sqrt(q)) ** 2
lam_minus = (1 - np.sqrt(q)) ** 2
print(f"MP band: [{lam_minus:.3f}, {lam_plus:.3f}]")
print(f"largest empirical eigenvalue: {w[0]:.3f}")
print(f"fraction above lam_+: {(w > lam_plus).mean():.2%}")

# Add a single planted factor by replacing one return direction
mkt = rng.standard_normal(T)
loadings = rng.uniform(0.5, 1.0, N)
Y = X + 3.0 * np.outer(mkt, loadings)
Y = (Y - Y.mean(0)) / Y.std(0)
C2 = (Y.T @ Y) / T
w2 = np.sort(np.linalg.eigvalsh(C2))[::-1]
print(f"with planted factor, largest eigenvalue: {w2[0]:.2f}  (towers above MP)")
fig, ax = plt.subplots(figsize=(8, 4))
component = np.arange(1, N + 1)
ax.plot(component, w, "o", ms=3.2, label="pure-noise eigenvalues")
ax.plot(1, w2[0], "o", ms=6.5, label="largest eigenvalue with planted factor")
ax.axhspan(lam_minus, lam_plus, color="#8da0cb", alpha=.18,
           label="Marchenko–Pastur noise band")
ax.axhline(lam_plus, color="#52658f", lw=1.2)
ax.set(xlabel="ordered component", ylabel="sample eigenvalue", yscale="log",
       title="Demo 7: noise stays in the benchmark band; the planted factor separates")
ax.legend(frameon=False)
ax.grid(alpha=.18, which="both")
fig.tight_layout()
save_lecture_figure(fig, "demo7_marchenko_pastur.png")
plt.show()
# Expected: pure-noise eigenvalues fall inside the MP band [0.23, 2.30]; a planted
#           factor produces an eigenvalue ~66, far outside it.

### Mini-project: audit a PCA representation

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run four checks in the notebook:

1. **Demo 5:** interpret the scree plot, PC1 loadings, and market-proxy correlation.
2. **Demo 6:** change only later covariance and test whether protected earlier scores move.
3. **Demo 7:** compare pure-noise eigenvalues and a planted factor with the Marchenko--Pastur band.
4. **Real-data companion:** fit PCA before the industry holdout and interpret frozen-test PC1.

**Notebook:** run **Demos 5–7** and the **real-data PCA companion**.

**Research record:** the simulated and industry PC1 interpretations; the noise-band comparison; the cutoff and largest earlier-score change under the frozen and global fits.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 8 — Cluster a correlation matrix to recover sectors

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
n_per = 12; n_sec = 4; T = 800
demo8_rng = np.random.default_rng(42)
sector_factors = demo8_rng.standard_normal((T, n_sec)) * 0.01
market = demo8_rng.standard_normal(T) * 0.008
ret = np.zeros((T, n_per * n_sec))
sectors = np.repeat(np.arange(n_sec), n_per)
for i, s in enumerate(sectors):
    ret[:, i] = 0.6 * market + 1.2 * sector_factors[:, s] + 0.01 * demo8_rng.standard_normal(T)
# scramble columns
perm = demo8_rng.permutation(ret.shape[1])
ret = ret[:, perm]
true_labels = sectors[perm]

corr = np.corrcoef(ret.T)
dist = np.sqrt(2 * np.clip(1 - corr, 0, None))
np.fill_diagonal(dist, 0)
clust = AgglomerativeClustering(n_clusters=4, metric="precomputed", linkage="average").fit(dist)

# purity
purity = 0
for k in range(4):
    members = np.where(clust.labels_ == k)[0]
    if len(members):
        most_common = np.bincount(true_labels[members]).max()
        purity += most_common
purity = purity / len(true_labels)
print(f"Cluster purity: {purity:.1%}")

order = np.argsort(clust.labels_)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
image = axes[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Raw (scrambled) correlation")
axes[1].imshow(corr[order][:, order], cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("Reordered by recovered clusters")
for ax in axes:
    ax.set_xlabel("asset order")
    ax.set_ylabel("asset order")
fig.suptitle(f"Demo 8: four planted sectors recovered (purity {purity:.0%})")
fig.subplots_adjust(top=.84, right=.88, wspace=.24)
color_axis = fig.add_axes([.91, .20, .018, .55])
fig.colorbar(image, cax=color_axis, label="correlation")
save_lecture_figure(fig, "demo8_sector_recovery.png")
plt.show()
# Expected: cluster purity ~100% on clean simulated sectors. Real correlation matrices
#           do not behave this well.

### Demo 9 — Frozen-centroid regime assignment

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
r = sim.regime_switching(n=1500, sig_calm=0.006, sig_turb=0.02,
                         p_stay_calm=0.985, p_stay_turb=0.96, seed=11)
r = np.asarray(r)

window = 21
series = pd.Series(r)
features = pd.DataFrame({
    "lagged_mean": series.rolling(window).mean().shift(1),
    "lagged_vol": series.rolling(window).std().shift(1),
}).dropna()
cutoff = 900
train = features.loc[features.index < cutoff]
test = features.loc[features.index >= cutoff]

train_scaler = StandardScaler().fit(train)
frozen_model = KMeans(n_clusters=2, n_init=20, random_state=11).fit(
    train_scaler.transform(train)
)
frozen_raw_centroids = train_scaler.inverse_transform(frozen_model.cluster_centers_)
frozen_high = np.argmax(frozen_raw_centroids[:, 1])
frozen_labels = (frozen_model.predict(train_scaler.transform(test)) == frozen_high).astype(int)

full_scaler = StandardScaler().fit(features)
full_model = KMeans(n_clusters=2, n_init=20, random_state=11).fit(
    full_scaler.transform(features)
)
full_raw_centroids = full_scaler.inverse_transform(full_model.cluster_centers_)
full_high = np.argmax(full_raw_centroids[:, 1])
retrospective_labels = (
    full_model.predict(full_scaler.transform(test)) == full_high
).astype(int)

disagree = np.mean(frozen_labels != retrospective_labels)
transitions = np.sum(frozen_labels[1:] != frozen_labels[:-1])
print("training cutoff index:", cutoff)
print("frozen centroids [lagged mean, lagged vol]:\n", frozen_raw_centroids.round(4))
print(f"test-label disagreement with full-sample clustering: {disagree:.1%}")
print(f"transitions under the frozen rule: {transitions}")

fig, axes = plt.subplots(2, 1, figsize=(10, 4.8), sharex=True)
axes[0].plot(test.index, test["lagged_vol"], color="black", lw=.8)
axes[0].set(ylabel="lagged volatility", title="Both label series describe the same later rows")
axes[1].step(test.index, frozen_labels, where="post", label="frozen-centroid label")
axes[1].step(test.index, retrospective_labels, where="post", alpha=.7,
             label="full-sample label")
axes[1].fill_between(
    test.index, 0, 1, where=frozen_labels != retrospective_labels,
    step="post", color="#d95f02", alpha=.28, label="labels disagree",
)
axes[1].set(xlabel="time index", ylabel="high-vol label", yticks=[0, 1])
axes[1].legend(frameon=False, ncol=2)
fig.suptitle(f"Demo 9: frozen and full-sample labels disagree on {disagree:.1%} of later dates")
fig.tight_layout(rect=(0, 0, 1, .94))
save_lecture_figure(fig, "demo9_frozen_centroid_assignments.png")
plt.show()

### Mini-project: recover groups, then make a regime prospective

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

1. **Demo 8:** cluster the scrambled correlation matrix and record sector recovery.
2. **Demo 9:** freeze earlier centroids, assign later dates, and compare with
   full-sample labels.

**Notebook:** run **Demos 8 and 9**.

**Research record:** Demo 8 cluster purity and reordered heatmap; Demo 9 training cutoff, frozen centroids, label disagreement, and transition timing.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Mini-project: make a regime prospective

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Cluster an earlier window, freeze the centroids, and assign the next
window one observation at a time. Compare with labels from a
full-sample clustering.

**Notebook:** run **Demo 9 — Frozen-centroid regime assignment**.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

## Where this leaves us

Optimization can converge perfectly to an unstable answer, and an attractive component or cluster can still be sample-dependent. These tools simplify or organize evidence; they do not interpret themselves and do not automatically create a decision rule.

Week 8 puts estimated signals into portfolio decisions and then studies how neural networks build flexible representations from data.